MySecretGraph2026!

In [76]:
import json
import urllib.request
from typing import TypedDict, List, Dict
from neo4j import GraphDatabase
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from pydantic import BaseModel ,Field
from langchain_core.prompts import ChatPromptTemplate
import os
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

if "GROQ_API_KEY" in os.environ:
    print('groq loaded')
else:
    print(f"failed with env api's")

groq loaded


In [77]:
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "MySecretGraph2026!"

In [78]:
def save_to_neo4j(triples: List[Dict[str, str]]):
    """Executes production Cypher queries to merge entities and connections safely."""
    query = """
    MERGE (s:Entity {name: $subject})
    MERGE (o:Entity {name: $object})
    WITH s, o
    CALL apoc.create.relationship(s, $predicate, {}, o) YIELD rel
    RETURN rel
    """
    # Alternative standard fallback if APOC is not installed out-of-the-box:
    fallback_query = """
    MERGE (s:Entity {name: $subject})
    MERGE (o:Entity {name: $object})
    WITH s, o
    MATCH (s), (o)
    CREATE (s)-[r:RELATION {type: $predicate}]->(o)
    """
    with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD)) as driver:
        with driver.session() as session:
            for t in triples:
                # Clean up formatting for valid Cypher relationship types
                rel_type = t['predicate'].strip().upper().replace(" ", "_")
                sub = t['subject'].strip().title()
                obj = t['object'].strip().title()
                
                # Using standard Cypher execution for immediate compatibility
                session.run(
                    f"MERGE (s:Entity {{name: $sub}}) "
                    f"MERGE (o:Entity {{name: $obj}}) "
                    f"MERGE (s)-[:{rel_type}]->(o)", 
                    sub=sub, obj=obj
                )

In [79]:
class GraphState(TypedDict):
    raw_text:str
    extracted_triples: List[Dict[str,str]]
    target_search: Dict[str,str]
    reasoning_result: str 
    

In [80]:
llm = ChatGroq(
    model="qwen/qwen3.6-27b",  # Crucial for stable .with_structured_output()
    temperature=0.1,                 # Dropped to 0.1 for maximum factual tracking precision
    max_retries=3
)

In [ ]:

# prompt = ChatPromptTemplate.from_messages([
#         ("human", "this is just a test to check if the api is working or not , just say hi ")
# ])
    
# # Run the structured pipeline
# chain = prompt | llm
# response = chain.invoke({})
# response

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "this is just a test to check if the api is working or not , just say hi"\n   - Key request: "just say hi"\n   - Context: Testing API functionality\n\n2.  **Identify Core Requirement:**\n   - The user explicitly wants a simple response: "hi"\n   - No additional information, explanations, or formatting needed.\n\n3.  **Formulate Response:**\n   - Keep it exactly as requested: "Hi!" or "hi"\n   - Match the tone: casual, direct\n   - Ensure it\'s just the requested word\n\n4.  **Final Output Generation:**\n   - "Hi!" (or just "hi")\n   - I\'ll go with "Hi!" as it\'s natural and matches the request.✅\n</think>\n\nHi!', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 190, 'prompt_tokens': 28, 'total_tokens': 218, 'completion_time': 0.372686305, 'completion_tokens_details': None, 'prompt_time': 0.002661487, 'prompt_tokens_details': None, 'queue_time': 0.1117

In [81]:
class Triple(BaseModel):
    subject: str = Field(description="The main entity or noun of the fact.")
    predicate: str = Field(description="The relationship or action verb linking the subject and object.")
    object: str = Field(description="The target entity, concept, or value.")


In [82]:
class KnowledgeGraph(BaseModel):
    triples: List[Triple] = Field(description="A list of all extracted factual triples.")

# 2. Bind the schema directly to your Groq LLM instance
structured_llm = llm.with_structured_output(KnowledgeGraph)

In [83]:

def extract_node(state: GraphState) -> Dict:
    """Node 1: Leverages qwen3.6-27b via Groq to guarantee perfectly structured triple extraction."""
    print("🤖 [Node: Extraction] Parsing raw text ...")
    
    system_prompt = (
        "You are an advanced Knowledge Graph extraction agent. Your job is to extract "
        "EVERY single factual relationship present in the text block. Do not skip details. "
        "Exhaustively pull all connections."
    )
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "Extract all facts from this text: {text}")
    ])
    
    try:
        # Run the structured pipeline
        chain = prompt | structured_llm
        response = chain.invoke({"text": state['raw_text']})
        
        # Pydantic guarantees response.triples is a valid list of objects matching our schema
        # We convert it back to a standard Python dictionary for LangGraph's state
        raw_triples = [triple.model_dump() for triple in response.triples]
        
        return {"extracted_triples": raw_triples}
    except Exception as e:
        print(f"💥 High-tier Extraction Error: {e}")
        return {"extracted_triples": []}

In [84]:
def store_node(state: GraphState) -> Dict:
    """Node 2: Persists the extracted variables directly into Neo4j."""
    triples = state.get('extracted_triples', [])
    
    # Double-check that we are actually dealing with data
    if not triples or not isinstance(triples, list):
        print("⚠️ [Node: Storage] No valid triples found to store.")
        return {}
        
    print(f"🗄️ [Node: Storage] Ingesting {len(triples)} triples into Neo4j...")
    save_to_neo4j(triples)
    return {}


In [85]:

def reason_node(state: GraphState) -> Dict:
    """Node 3: Leverages Neo4j's optimized structural tracking to query paths instantly."""
    print("🧠 [Node: Reasoning] Querying Neo4j production database index...")
    start = state['target_search'].get('start').title()
    end = state['target_search'].get('end').title()
    
    # Powerful Cypher pattern matching engine replaces our manual python BFS/DFS loops
    cypher_query = """
    MATCH path = shortestPath((s:Entity {name: $start})-[*..5]->(e:Entity {name: $end}))
    RETURN path
    """
    
    with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD)) as driver:
        with driver.session() as session:
            result = session.run(cypher_query, start=start, end=end)
            record = result.single()
            if record:
                # Production path formatting
                nodes = [node['name'] for node in record['path'].nodes]
                return {"reasoning_result": f"✅ Proof found via Neo4j: " + " -> ".join(nodes)}
            
    return {"reasoning_result": f"❌ Neo4j could not map a structural path from {start} to {end}."}


In [86]:

# --- 4. BUILD THE AGENTIC STATE GRAPH SYSTEM ---
workflow = StateGraph(GraphState)

# Define our structured execution nodes
workflow.add_node("extraction_agent", extract_node)
workflow.add_node("storage_agent", store_node)
workflow.add_node("reasoning_agent", reason_node)

# Map explicit pipeline sequence links
workflow.set_entry_point("extraction_agent")
workflow.add_edge("extraction_agent", "storage_agent")
workflow.add_edge("storage_agent", "reasoning_agent")
workflow.add_edge("reasoning_agent", END)

# Compile into a production system runtime executable
app = workflow.compile()

In [87]:
input_payload = {
        "raw_text": "Autogen is framework engineered by Microsoft. Microsoft is headquartered in Redmond. Redmond is in Washington state.",
        "target_search": {"start": "Autogen", "end": "Washington State"}
    }
    
print("🚀 Starting Production Knowledge Graph Pipeline Engine...")
final_output = app.invoke(input_payload)

print("\n🏁 [Pipeline Complete Result]")
print(final_output["reasoning_result"])


🚀 Starting Production Knowledge Graph Pipeline Engine...
🤖 [Node: Extraction] Parsing raw text ...
🗄️ [Node: Storage] Ingesting 3 triples into Neo4j...
🧠 [Node: Reasoning] Querying Neo4j production database index...

🏁 [Pipeline Complete Result]
✅ Proof found via Neo4j: Autogen -> Microsoft -> Redmond -> Washington State


In [58]:
print(final_output["extracted_triples"])

[]
